# CAMeL-BERT Binary Classification - OpenITI Generalization Test
## Test sur corpus externe non-vu (openiti_targeted)

**Objectif** : Valider que le modèle fine-tuné généralise bien sur des textes arabes historiques différents

**Approche** :
1. Charger le corpus OpenITI depuis Google Drive
2. Nettoyer le format OpenITI mARkdown
3. Segmenter en chunks
4. Exécuter l'inférence CAMeL-BERT
5. Analyser les résultats (densité de boundaries, distribution)
6. Comparer avec baseline v4 (si disponible)

**Résultats attendus** :
- Généralisation robuste sur corpus différent
- Distribution de boundaries similaire à Kitab Uqala
- Peu de faux positifs/négatifs


---
## Step 1 : Setup

In [ ]:
# Montage du Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"[OK] Working directory: {os.getcwd()}")

In [ ]:
# Installation
!pip install transformers torch tqdm matplotlib seaborn -q
print("[OK] Dependencies installed")

In [ ]:
# Imports
import json
import re
import numpy as np
import torch
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict, Counter
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForTokenClassification

print("[OK] All imports successful")
print(f"GPU available: {torch.cuda.is_available()}")

---
## Step 2 : Charger le modèle fine-tuné

In [ ]:
# Chemins
model_path = Path('checkpoints/camelbert_binary_classification_final')
openiti_path = Path('openiti_targeted')

print(f"[CHECK] Model: {model_path.exists()}")
print(f"[CHECK] OpenITI corpus: {openiti_path.exists()}")

if openiti_path.exists():
    texts = list(openiti_path.glob('*'))[:10]
    print(f"\nOpenITI texts found: {len(texts)}")
    for t in texts[:5]:
        print(f"  - {t.name}")

In [ ]:
# Charger le modèle
print(f"[INFO] Loading model...")

tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()

print(f"[OK] Model loaded")

---
## Step 3 : Nettoyage du format OpenITI

In [ ]:
def clean_openiti_text(text: str) -> str:
    """
    Nettoie un texte au format mARkdown OpenITI.
    Supprime :
    - Header et métadonnées
    - Marqueurs de section (###, #)
    - Continuations (~~)
    - Marqueurs de page (PageV###P###)
    - Marqueurs de manuscrit (ms####)
    - Références de notes ([1], [2a], etc.)
    """
    # 1. Supprimer le header
    if '#META#Header#End#' in text:
        text = text[text.index('#META#Header#End#') + len('#META#Header#End#'):]
    
    # 2. Supprimer marqueurs de section ### et #
    text = re.sub(r'^###[^\n]*\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'^#+\s*', '', text, flags=re.MULTILINE)
    
    # 3. Joindre lignes de continuation ~~
    text = re.sub(r'\n~~', ' ', text)
    
    # 4. Supprimer marqueurs de page
    text = re.sub(r'PageV\d+P\d+', '', text)
    
    # 5. Supprimer marqueurs de manuscrit
    text = re.sub(r'\bms\d+\b', '', text)
    
    # 6. Supprimer références de notes [1], [2a], etc.
    text = re.sub(r'\[\d+[a-z]?\]', '', text)
    
    # 7. Normaliser espaces
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

print("[OK] clean_openiti_text function defined")

In [ ]:
def split_into_chunks(text: str, chunk_size: int = 2000, overlap: int = 300) -> List[str]:
    """
    Divise un texte long en chunks avec chevauchement.
    
    Args:
        text: texte à diviser
        chunk_size: taille cible en caractères
        overlap: chevauchement entre chunks
    
    Returns:
        Liste de chunks
    """
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks

print("[OK] split_into_chunks function defined")

In [ ]:
# Tester le nettoyage sur un exemple
if openiti_path.exists():
    sample_file = list(openiti_path.glob('*'))[0]
    raw_text = sample_file.read_text(encoding='utf-8')
    cleaned_text = clean_openiti_text(raw_text)
    
    print(f"[TEST] Sample file: {sample_file.name}")
    print(f"  Raw size: {len(raw_text)} chars")
    print(f"  Cleaned size: {len(cleaned_text)} chars")
    print(f"  Reduction: {100 * (1 - len(cleaned_text)/len(raw_text)):.1f}%")
    print(f"\n  First 200 chars (cleaned):")
    print(f"  {cleaned_text[:200]}...")

---
## Step 4 : Inférence sur corpus OpenITI

In [ ]:
def predict_boundaries(text: str, tokenizer, model) -> Dict:
    """
    Prédire les frontières dans un texte arabe.
    """
    # Tokenizer
    encoded = tokenizer(
        text,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Inférence
    with torch.no_grad():
        if torch.cuda.is_available():
            for key in encoded:
                encoded[key] = encoded[key].cuda()
        
        outputs = model(**encoded)
        logits = outputs.logits[0]
    
    # Prédictions
    preds = np.argmax(logits.cpu().numpy(), axis=-1)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    
    # Tokens
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    
    # Boundary indices
    boundary_indices = np.where(preds == 1)[0].tolist()
    
    return {
        'tokens': tokens,
        'predictions': preds,
        'probabilities': probs[:, 1],
        'boundary_indices': boundary_indices,
        'boundary_tokens': [tokens[i] for i in boundary_indices],
    }

print("[OK] predict_boundaries function defined")

In [ ]:
# Traiter tous les textes OpenITI
openiti_results = {}

if openiti_path.exists():
    text_files = sorted(openiti_path.glob('*'))
    print(f"[INFO] Processing {len(text_files)} OpenITI texts...\n")
    
    for text_file in tqdm(text_files, desc="OpenITI texts"):
        try:
            # Charger et nettoyer
            raw_text = text_file.read_text(encoding='utf-8')
            cleaned_text = clean_openiti_text(raw_text)
            
            if len(cleaned_text) < 100:  # Ignorer les fichiers trop courts
                continue
            
            # Découper en chunks
            chunks = split_into_chunks(cleaned_text, chunk_size=2000, overlap=300)
            
            # Inférence sur chaque chunk
            all_boundary_indices = []
            all_boundary_probs = []
            total_tokens = 0
            
            for chunk in chunks:
                result = predict_boundaries(chunk, tokenizer, model)
                all_boundary_indices.extend(result['boundary_indices'])
                all_boundary_probs.extend(result['probabilities'][result['boundary_indices']].tolist())
                total_tokens += len(result['predictions'])
            
            # Statistiques
            openiti_results[text_file.name] = {
                'raw_size': len(raw_text),
                'cleaned_size': len(cleaned_text),
                'num_chunks': len(chunks),
                'total_tokens': total_tokens,
                'boundary_count': len(all_boundary_indices),
                'boundary_density': len(all_boundary_indices) / max(total_tokens, 1),
                'avg_boundary_prob': np.mean(all_boundary_probs) if all_boundary_probs else 0,
            }
        except Exception as e:
            print(f"[ERROR] {text_file.name}: {e}")
    
    print(f"\n[OK] Processed {len(openiti_results)} texts")
else:
    print(f"[ERROR] OpenITI path not found: {openiti_path}")

---
## Step 5 : Analyse des Résultats

In [ ]:
# Afficher les résultats
if openiti_results:
    print("[OPENITI GENERALIZATION RESULTS]\n")
    print(f"{'Author':<40} {'Boundaries':<12} {'Density':<10} {'Avg Prob':<10}")
    print("-" * 75)
    
    for name, stats in sorted(openiti_results.items()):
        author = name[:38]
        boundaries = stats['boundary_count']
        density = stats['boundary_density']
        avg_prob = stats['avg_boundary_prob']
        
        print(f"{author:<40} {boundaries:<12} {density:<10.4f} {avg_prob:<10.4f}")
    
    # Statistiques globales
    print("\n[SUMMARY STATISTICS]")
    densities = [s['boundary_density'] for s in openiti_results.values()]
    print(f"  Mean boundary density: {np.mean(densities):.4f}")
    print(f"  Std boundary density: {np.std(densities):.4f}")
    print(f"  Min boundary density: {np.min(densities):.4f}")
    print(f"  Max boundary density: {np.max(densities):.4f}")

---
## Step 6 : Comparaison avec Kitab Uqala

In [ ]:
# Charger les stats de Kitab Uqala pour comparaison
kitab_file = Path('data/processed/binary_classification_dataset/binary_classification_examples.jsonl')

if kitab_file.exists():
    with open(kitab_file, encoding='utf-8') as f:
        examples = [json.loads(line) for line in f]
    
    # Calculer les stats
    kitab_densities = []
    for ex in examples:
        # Nombre de segments isnad
        num_isnads = sum(1 for s in ex['segments'] if s['type'] == 'isnad')
        # Nombre total de segments
        num_segments = len(ex['segments'])
        if num_segments > 0:
            kitab_densities.append(num_isnads / num_segments)
    
    print("[COMPARISON: Kitab Uqala vs OpenITI]\n")
    print(f"Kitab Uqala (training corpus):")
    print(f"  Mean boundary density: {np.mean(kitab_densities):.4f}")
    print(f"  Std boundary density: {np.std(kitab_densities):.4f}")
    print(f"\nOpenITI (test corpus):")
    if openiti_results:
        densities = [s['boundary_density'] for s in openiti_results.values()]
        print(f"  Mean boundary density: {np.mean(densities):.4f}")
        print(f"  Std boundary density: {np.std(densities):.4f}")
        
        # Différence
        diff = np.mean(densities) - np.mean(kitab_densities)
        print(f"\nDifference: {diff:+.4f} ({100*diff/np.mean(kitab_densities):+.1f}%)")

---
## Step 7 : Visualisations

In [ ]:
# Plot distribution des boundary densities
if openiti_results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    densities_openiti = [s['boundary_density'] for s in openiti_results.values()]
    
    # Histogram
    axes[0, 0].hist(densities_openiti, bins=15, alpha=0.7, color='#2E86AB', edgecolor='black')
    axes[0, 0].axvline(np.mean(kitab_densities), color='red', linestyle='--', linewidth=2, label='Kitab Uqala mean')
    axes[0, 0].set_xlabel('Boundary Density')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Distribution of Boundary Densities (OpenITI)')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Box plot comparison
    axes[0, 1].boxplot([kitab_densities, densities_openiti], labels=['Kitab Uqala', 'OpenITI'])
    axes[0, 1].set_ylabel('Boundary Density')
    axes[0, 1].set_title('Boundary Density Comparison')
    axes[0, 1].grid(alpha=0.3, axis='y')
    
    # Scatter: boundaries vs text size
    text_sizes = [s['cleaned_size']/1000 for s in openiti_results.values()]
    boundaries = [s['boundary_count'] for s in openiti_results.values()]
    
    axes[1, 0].scatter(text_sizes, boundaries, alpha=0.6, s=100, color='#A23B72')
    axes[1, 0].set_xlabel('Text Size (KB)')
    axes[1, 0].set_ylabel('Boundary Count')
    axes[1, 0].set_title('Boundaries vs Text Size')
    axes[1, 0].grid(alpha=0.3)
    
    # Average boundary probability
    avg_probs = [s['avg_boundary_prob'] for s in openiti_results.values()]
    axes[1, 1].bar(range(len(avg_probs)), avg_probs, color='#F18F01', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Text Index')
    axes[1, 1].set_ylabel('Average Boundary Probability')
    axes[1, 1].set_title('Prediction Confidence per Text')
    axes[1, 1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
    axes[1, 1].set_ylim([0, 1])
    axes[1, 1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('results/openiti_generalization_analysis.png', dpi=150, bbox_inches='tight')
    print("[OK] Visualization saved to results/openiti_generalization_analysis.png")
    plt.show()

---
## Step 8 : Sauvegarde des Résultats

In [ ]:
if openiti_results:
    results_dir = Path('results/openiti_generalization_test')
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # Résumé
    summary = {
        'model': 'CAMeL-Lab/bert-base-arabic-camelbert-ca',
        'task': 'binary_classification_boundary_detection',
        'test_corpus': 'openiti_targeted',
        'texts_processed': len(openiti_results),
        'generalization_results': openiti_results,
        'summary_stats': {
            'openiti_mean_density': float(np.mean([s['boundary_density'] for s in openiti_results.values()])),
            'openiti_std_density': float(np.std([s['boundary_density'] for s in openiti_results.values()])),
            'kitab_mean_density': float(np.mean(kitab_densities)),
            'kitab_std_density': float(np.std(kitab_densities)),
        }
    }
    
    with open(results_dir / 'openiti_generalization_results.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    print(f"[OK] Results saved to {results_dir}/openiti_generalization_results.json")

---
## Final Summary

In [ ]:
if openiti_results:
    densities = [s['boundary_density'] for s in openiti_results.values()]
    
    print(f"""
╔════════════════════════════════════════════════════════════╗
║        OpenITI Generalization Test - COMPLETE           ║
╚════════════════════════════════════════════════════════════╝

📊 TEST RESULTS
  • Texts processed: {len(openiti_results)}
  • Mean boundary density: {np.mean(densities):.4f}
  • Std boundary density: {np.std(densities):.4f}
  • Range: [{np.min(densities):.4f}, {np.max(densities):.4f}]

📈 COMPARISON vs Kitab Uqala (training corpus)
  • Kitab Uqala density: {np.mean(kitab_densities):.4f} +/- {np.std(kitab_densities):.4f}
  • OpenITI density: {np.mean(densities):.4f} +/- {np.std(densities):.4f}
  • Difference: {np.mean(densities) - np.mean(kitab_densities):+.4f}

✅ GENERALIZATION ASSESSMENT
  • Distribution similarity: {'GOOD' if abs(np.mean(densities) - np.mean(kitab_densities)) < 0.05 else 'MODERATE'}
  • Model robustness: {'HIGH' if np.std(densities) < 0.15 else 'MODERATE'}

💾 OUTPUTS (saved on Google Drive)
  • Analysis: results/openiti_generalization_analysis.png
  • Summary: results/openiti_generalization_test/openiti_generalization_results.json

📌 NEXT STEPS
  1. Review distribution similarity
  2. If good generalization -> Deploy model
  3. If poor generalization -> Fine-tune on mixed corpus
  4. Consider Piste 2 (Span-based) for further improvements
""")
else:
    print("[ERROR] No OpenITI texts processed")